
# ENEMDU 2025 — Notebook robusto de métricas avanzadas multiclase
## Reproduce los 3 clasificadores del artículo y añade evaluación avanzada

Este notebook está diseñado específicamente para el ZIP completo del proyecto:

`enedmu_adultos_mayores.zip`

### Ventaja de esta versión

**No depende de que existan `modelo_logit.joblib` y `modelo_rf.joblib`.**

El notebook:

1. abre el ZIP completo;
2. localiza automáticamente `adultos_mayores_con_cluster.csv`;
3. reconstruye exactamente las variables de `05_supervisado.py`;
4. reutiliza `split_test_idx.npy` si existe;
5. reentrena:
   - Regresión Logística multinomial;
   - Random Forest;
   - XGBoost;
6. reproduce las métricas originales;
7. agrega:
   - Brier Score multiclase;
   - calibración;
   - Platt;
   - Isotonic;
   - ROC one-vs-rest;
   - Precision–Recall one-vs-rest;
   - Average Precision;
   - Precision / Recall / F1 por segmento;
   - falsos negativos y destino de errores;
   - threshold específico del segmento 2;
   - escenario Recall(C2) ≥ 70%;
   - métricas ponderadas por `fexp` como análisis suplementario;
8. exporta todo a un ZIP.

---

## Segmentos

1. **Jubilados sin actividad laboral**
2. **Trabajadores de mayores ingresos**
3. **Trabajadores de menores ingresos**
4. **Inactivos sin pensión ni empleo**

---

## Importante

La clasificación multiclase principal continúa siendo:

\[
\widehat C=\arg\max_k P(C_k)
\]

El análisis de threshold se aplica **solo al segmento 2 como prueba de sensibilidad**.

El TEST original del artículo no se utiliza para seleccionar calibración ni threshold.


## 1. Instalar e importar librerías

In [ ]:

%pip install -q xgboost joblib

import os
import json
import shutil
import warnings
import time
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from scipy.special import logit

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.preprocessing import label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    log_loss,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)

from xgboost import XGBClassifier

RANDOM_STATE = 42
CLASES = np.array([1, 2, 3, 4])

NOMBRES = {
    1: "Jubilados sin actividad laboral",
    2: "Trabajadores de mayores ingresos",
    3: "Trabajadores de menores ingresos",
    4: "Inactivos sin pensión ni empleo",
}

# 5 folds para probabilidades out-of-fold usadas en calibración.
N_FOLDS_OOF = 5

# Si Platt/Isotonic mejora menos que esto frente a RAW,
# conservamos RAW por parsimonia.
MIN_MEJORA_BRIER = 1e-4

RECALL_OBJETIVO_C2 = 0.70

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Entorno listo.")



## 2. Subir el ZIP completo

En esta celda sube exactamente el archivo que ya tienes:

**`enedmu_adultos_mayores.zip`**

No tienes que preparar un ZIP distinto.


In [ ]:

from google.colab import files

uploaded = files.upload()

zip_candidates = [
    nombre
    for nombre in uploaded.keys()
    if nombre.lower().endswith(".zip")
]

if not zip_candidates:
    raise FileNotFoundError(
        "Debes subir el ZIP completo del proyecto."
    )

ZIP_PATH = zip_candidates[0]

PROYECTO = Path("/content/enemdu_proyecto")

if PROYECTO.exists():
    shutil.rmtree(PROYECTO)

PROYECTO.mkdir(
    parents=True,
    exist_ok=True
)

shutil.unpack_archive(
    ZIP_PATH,
    PROYECTO
)

print("ZIP:", ZIP_PATH)
print("Descomprimido en:", PROYECTO)


## 3. Detectar automáticamente los archivos importantes

In [ ]:

def buscar_archivo(nombre, obligatorio=True):
    encontrados = list(
        PROYECTO.rglob(nombre)
    )

    if not encontrados:
        if obligatorio:
            raise FileNotFoundError(
                f"No se encontró {nombre} dentro del ZIP."
            )
        return None

    return encontrados[0]


ARCHIVO_CLUSTER = buscar_archivo(
    "adultos_mayores_con_cluster.csv"
)

SPLIT_FILE = buscar_archivo(
    "split_test_idx.npy",
    obligatorio=False
)

METRICAS_LOGIT_ORIG = buscar_archivo(
    "metricas_logit.csv",
    obligatorio=False
)

METRICAS_RF_ORIG = buscar_archivo(
    "metricas_rf.csv",
    obligatorio=False
)

METRICAS_XGB_ORIG = buscar_archivo(
    "metricas_xgb.csv",
    obligatorio=False
)

SHAP_NPZ = buscar_archivo(
    "shap_values_xgb.npz",
    obligatorio=False
)

RESULTADOS = Path(
    "/content/resultados_avanzados_multiclase"
)

TAB = RESULTADOS / "tablas"
FIG = RESULTADOS / "figuras"
MOD = RESULTADOS / "modelos"

for carpeta in [TAB, FIG, MOD]:
    carpeta.mkdir(
        parents=True,
        exist_ok=True
    )

print("Base con clusters:", ARCHIVO_CLUSTER)
print("Split original:", SPLIT_FILE)
print("SHAP existente:", SHAP_NPZ)
print("Resultados nuevos:", RESULTADOS)



## 4. Reconstruir exactamente los predictores de `05_supervisado.py`

Los predictores continúan siendo estrictamente sociodemográficos:

- sexo;
- edad;
- jefatura;
- educación;
- área;
- etnia;
- provincia.

**No se introducen ingreso, empleo, pensión ni monto de pensión en la fase supervisada.**


In [ ]:

def construir_features(path_csv):

    df = pd.read_csv(
        path_csv
    )

    requeridas = [
        "cluster",
        "p02",
        "p03",
        "p04",
        "p10a_edu",
        "area",
        "prov",
        "p15",
        "fexp",
    ]

    faltantes = [
        c
        for c in requeridas
        if c not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"Faltan columnas: {faltantes}"
        )

    df = df.dropna(
        subset=[
            "p02",
            "p03",
            "p04",
            "p10a_edu",
            "area",
            "prov",
            "p15",
        ]
    ).reset_index(drop=True)

    y = (
        df["cluster"]
        .astype(int)
        .values
    )

    X = pd.DataFrame(
        index=df.index
    )

    X["sexo_mujer"] = (
        df["p02"] == 2
    ).astype(int)

    X["edad"] = (
        df["p03"]
        .astype(float)
    )

    X["jefe_hogar"] = (
        df["p04"] == 1
    ).astype(int)

    X["educ"] = (
        df["p10a_edu"]
        .astype(float)
    )

    X["area_rural"] = (
        df["area"] == 2
    ).astype(int)

    etnia = pd.get_dummies(
        df["p15"].astype(str),
        prefix="etnia",
        dtype=int
    )

    prov = pd.get_dummies(
        df["prov"].astype(str),
        prefix="prov",
        dtype=int
    )

    X = pd.concat(
        [X, etnia, prov],
        axis=1
    )

    return df, X, y


df, X, y = construir_features(
    ARCHIVO_CLUSTER
)

print("N total:", len(df))
print("N features:", X.shape[1])

display(
    pd.Series(y)
    .value_counts()
    .sort_index()
    .rename("n")
    .to_frame()
)



## 5. Reutilizar el split 70/30 original

Si el ZIP contiene `split_test_idx.npy`, se usa exactamente ese split.

Si no existiera, se reconstruye con:

- `test_size=0.30`
- `stratify=y`
- `random_state=42`


In [ ]:

if SPLIT_FILE is not None:

    idx_test = np.load(
        SPLIT_FILE
    )

    print(
        "Usando split_test_idx.npy original."
    )

else:

    _, idx_test = train_test_split(
        np.arange(len(y)),
        test_size=0.30,
        stratify=y,
        random_state=RANDOM_STATE
    )

    print(
        "split_test_idx.npy no encontrado; "
        "split reproducido con semilla 42."
    )

idx_train = np.setdiff1d(
    np.arange(len(y)),
    idx_test
)

Xtr = X.iloc[
    idx_train
].copy()

Xte = X.iloc[
    idx_test
].copy()

ytr = y[
    idx_train
]

yte = y[
    idx_test
]

wte = (
    df.iloc[idx_test]["fexp"]
    .to_numpy(dtype=float)
)

print(
    f"Train = {len(Xtr):,}"
)

print(
    f"Test  = {len(Xte):,}"
)

print("\nDistribución TEST:")

display(
    pd.Series(yte)
    .value_counts()
    .sort_index()
    .rename("n")
    .to_frame()
)



# PARTE I — Reentrenar los tres clasificadores del artículo

## 6. Regresión Logística multinomial

Se conserva la especificación del script original:

- `solver='lbfgs'`
- `C=1`
- `class_weight='balanced'`
- `max_iter=2000`


In [ ]:

t0 = time.time()

modelo_logit = LogisticRegression(
    solver="lbfgs",
    max_iter=2000,
    C=1.0,
    class_weight="balanced",
)

modelo_logit.fit(
    Xtr,
    ytr
)

print(
    f"Logit entrenado en "
    f"{time.time()-t0:.1f} s"
)



## 7. Random Forest

Se reproduce la lógica del script original:

- 250 árboles;
- `class_weight='balanced'`;
- búsqueda de:
  - `max_depth`;
  - `min_samples_leaf`;
- 3 combinaciones;
- CV = 2;
- scoring = `f1_macro`;
- búsqueda sobre submuestra fija de hasta 12.000 observaciones.


In [ ]:

rng = np.random.RandomState(
    RANDOM_STATE
)

sub = rng.choice(
    len(Xtr),
    size=min(
        12000,
        len(Xtr)
    ),
    replace=False
)

rf_base = RandomForestClassifier(
    n_estimators=250,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_search = RandomizedSearchCV(
    rf_base,
    {
        "max_depth": [
            8,
            14,
            None
        ],
        "min_samples_leaf": [
            2,
            5,
            10
        ],
    },
    n_iter=3,
    cv=2,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

t0 = time.time()

rf_search.fit(
    Xtr.iloc[sub],
    ytr[sub]
)

modelo_rf = rf_search.best_estimator_

modelo_rf.fit(
    Xtr,
    ytr
)

print(
    "Mejores parámetros RF:",
    rf_search.best_params_
)

print(
    "CV F1 macro:",
    rf_search.best_score_
)

print(
    f"RF entrenado en "
    f"{time.time()-t0:.1f} s"
)



## 8. XGBoost multiclase

Se conserva:

- `objective='multi:softprob'`
- 350 árboles
- profundidad 6
- learning rate 0.05
- subsample 0.8
- colsample_bytree 0.8
- ponderación inversa por frecuencia de clase


In [ ]:

freq_train = (
    pd.Series(ytr)
    .value_counts(
        normalize=True
    )
    .to_dict()
)

sw_xgb = np.array([
    1.0 / freq_train[v]
    for v in ytr
])

modelo_xgb = XGBClassifier(
    objective="multi:softprob",
    n_estimators=350,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
    eval_metric="mlogloss",
)

t0 = time.time()

modelo_xgb.fit(
    Xtr,
    ytr - 1,
    sample_weight=sw_xgb
)

print(
    f"XGBoost entrenado en "
    f"{time.time()-t0:.1f} s"
)


## 9. Guardar los tres modelos reentrenados

In [ ]:

modelos = {
    "logit": modelo_logit,
    "rf": modelo_rf,
    "xgb": modelo_xgb,
}

for nombre, modelo in modelos.items():

    path = MOD / (
        f"modelo_{nombre}_reentrenado.joblib"
    )

    joblib.dump(
        modelo,
        path
    )

    print(
        "Guardado:",
        path.name
    )


## 10. Funciones para probabilidades y métricas

In [ ]:

def one_hot_y(y):
    return label_binarize(
        y,
        classes=CLASES
    )


def normalizar_probabilidades(p):

    p = np.asarray(
        p,
        dtype=float
    )

    p = np.clip(
        p,
        1e-12,
        1.0
    )

    s = p.sum(
        axis=1,
        keepdims=True
    )

    s[s == 0] = 1.0

    return p / s


def proba_modelo(
    nombre,
    modelo,
    Xs
):

    p = np.asarray(
        modelo.predict_proba(Xs),
        dtype=float
    )

    if nombre == "xgb":
        # XGB fue ajustado con clases 0..3.
        return normalizar_probabilidades(
            p
        )

    clases_mod = np.asarray(
        modelo.classes_,
        dtype=int
    )

    out = np.zeros(
        (len(Xs), 4)
    )

    for j, c in enumerate(CLASES):

        pos = np.where(
            clases_mod == c
        )[0]

        if len(pos) != 1:
            raise RuntimeError(
                f"No se encontró la clase {c} "
                f"en {nombre}."
            )

        out[:, j] = p[
            :,
            pos[0]
        ]

    return normalizar_probabilidades(
        out
    )


def pred_argmax(p):

    return CLASES[
        np.argmax(
            p,
            axis=1
        )
    ]


def brier_multiclase(
    y_true,
    proba,
    sample_weight=None
):

    Y = one_hot_y(
        y_true
    )

    err = np.sum(
        (proba - Y) ** 2,
        axis=1
    )

    if sample_weight is None:
        return float(
            np.mean(err)
        )

    return float(
        np.average(
            err,
            weights=sample_weight
        )
    )


def ece_confianza(
    y_true,
    proba,
    n_bins=10,
    sample_weight=None
):

    pred = pred_argmax(
        proba
    )

    conf = np.max(
        proba,
        axis=1
    )

    correcto = (
        pred == y_true
    ).astype(float)

    if sample_weight is None:
        pesos = np.ones(
            len(y_true)
        )
    else:
        pesos = np.asarray(
            sample_weight,
            dtype=float
        )

    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )

    total = pesos.sum()
    ece = 0.0

    for b in range(n_bins):

        if b == n_bins - 1:
            mask = (
                (conf >= edges[b])
                & (conf <= edges[b+1])
            )
        else:
            mask = (
                (conf >= edges[b])
                & (conf < edges[b+1])
            )

        if not np.any(mask):
            continue

        wb = pesos[
            mask
        ]

        conf_b = np.average(
            conf[mask],
            weights=wb
        )

        acc_b = np.average(
            correcto[mask],
            weights=wb
        )

        ece += (
            wb.sum() / total
        ) * abs(
            acc_b - conf_b
        )

    return float(ece)


def metricas_globales(
    y_true,
    proba,
    sample_weight=None
):

    pred = pred_argmax(
        proba
    )

    Y = one_hot_y(
        y_true
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            pred,
            sample_weight=sample_weight
        ),

        "precision_macro": precision_score(
            y_true,
            pred,
            labels=CLASES,
            average="macro",
            zero_division=0,
            sample_weight=sample_weight
        ),

        "recall_macro": recall_score(
            y_true,
            pred,
            labels=CLASES,
            average="macro",
            zero_division=0,
            sample_weight=sample_weight
        ),

        "f1_macro": f1_score(
            y_true,
            pred,
            labels=CLASES,
            average="macro",
            zero_division=0,
            sample_weight=sample_weight
        ),

        "f1_weighted": f1_score(
            y_true,
            pred,
            labels=CLASES,
            average="weighted",
            zero_division=0,
            sample_weight=sample_weight
        ),

        "kappa": cohen_kappa_score(
            y_true,
            pred,
            sample_weight=sample_weight
        ),

        "log_loss": log_loss(
            y_true,
            proba,
            labels=CLASES,
            sample_weight=sample_weight
        ),

        "brier_multiclase": brier_multiclase(
            y_true,
            proba,
            sample_weight=sample_weight
        ),

        "roc_auc_ovr_macro": roc_auc_score(
            y_true,
            proba,
            labels=CLASES,
            multi_class="ovr",
            average="macro",
            sample_weight=sample_weight
        ),

        "average_precision_macro": average_precision_score(
            Y,
            proba,
            average="macro",
            sample_weight=sample_weight
        ),

        "ece_10bins": ece_confianza(
            y_true,
            proba,
            sample_weight=sample_weight
        ),
    }



# PARTE II — Reproducción del artículo

## 11. Métricas originales y nuevas métricas RAW


In [ ]:

probas_test_raw = {}

filas_raw = []
filas_raw_pond = []

for nombre, modelo in modelos.items():

    p = proba_modelo(
        nombre,
        modelo,
        Xte
    )

    probas_test_raw[
        nombre
    ] = p

    m = metricas_globales(
        yte,
        p
    )

    m["modelo"] = nombre

    filas_raw.append(
        m
    )

    mp = metricas_globales(
        yte,
        p,
        sample_weight=wte
    )

    mp["modelo"] = nombre

    filas_raw_pond.append(
        mp
    )


metricas_raw = (
    pd.DataFrame(
        filas_raw
    )
    .set_index("modelo")
)

metricas_raw_ponderadas = (
    pd.DataFrame(
        filas_raw_pond
    )
    .set_index("modelo")
)

print(
    "MÉTRICAS PRINCIPALES — NO PONDERADAS"
)

display(
    metricas_raw
)

print(
    "MÉTRICAS SUPLEMENTARIAS — PONDERADAS fexp"
)

display(
    metricas_raw_ponderadas
)



## 12. Comprobar reproducibilidad contra los CSV originales

Si los CSV originales están dentro del ZIP, se comparan automáticamente.

Pequeñas diferencias pueden aparecer por versiones de scikit-learn/XGBoost, pero la estructura general debería conservarse.


In [ ]:

archivos_originales = {
    "logit": METRICAS_LOGIT_ORIG,
    "rf": METRICAS_RF_ORIG,
    "xgb": METRICAS_XGB_ORIG,
}

comparacion_original = []

for nombre, archivo in archivos_originales.items():

    if archivo is None:
        continue

    original = pd.read_csv(
        archivo
    ).iloc[0]

    actual = metricas_raw.loc[
        nombre
    ]

    for metrica_original, metrica_actual in [
        ("accuracy", "accuracy"),
        ("f1_macro", "f1_macro"),
        ("f1_weighted", "f1_weighted"),
        ("kappa", "kappa"),
        ("log_loss", "log_loss"),
    ]:

        comparacion_original.append({
            "modelo": nombre,
            "metrica": metrica_original,
            "original": float(
                original[
                    metrica_original
                ]
            ),
            "reproducido": float(
                actual[
                    metrica_actual
                ]
            ),
            "diferencia": float(
                actual[
                    metrica_actual
                ]
                - original[
                    metrica_original
                ]
            ),
        })


comparacion_original = pd.DataFrame(
    comparacion_original
)

display(
    comparacion_original
)


## 13. Precision, Recall, F1, ROC-AUC y AP por segmento

In [ ]:

def metricas_por_clase(
    y_true,
    proba,
    modelo
):

    pred = pred_argmax(
        proba
    )

    filas = []

    for j, c in enumerate(CLASES):

        ybin = (
            y_true == c
        ).astype(int)

        pbin = (
            pred == c
        ).astype(int)

        filas.append({
            "modelo": modelo,
            "clase": c,
            "segmento": NOMBRES[c],

            "precision": precision_score(
                ybin,
                pbin,
                zero_division=0
            ),

            "recall": recall_score(
                ybin,
                pbin,
                zero_division=0
            ),

            "f1": f1_score(
                ybin,
                pbin,
                zero_division=0
            ),

            "roc_auc_ovr": roc_auc_score(
                ybin,
                proba[:, j]
            ),

            "average_precision_ovr": (
                average_precision_score(
                    ybin,
                    proba[:, j]
                )
            ),

            "soporte": int(
                ybin.sum()
            ),

            "prevalencia": float(
                ybin.mean()
            ),
        })

    return pd.DataFrame(
        filas
    )


metricas_por_segmento = pd.concat(
    [
        metricas_por_clase(
            yte,
            probas_test_raw[nombre],
            nombre
        )
        for nombre in modelos
    ],
    ignore_index=True
)

display(
    metricas_por_segmento
)



# PARTE III — Calibración sin tocar TEST

## 14. Crear probabilidades out-of-fold dentro de TRAIN

Se utilizan 5 folds estratificados.

Cada observación recibe probabilidades producidas por un modelo que no fue ajustado con esa observación.


In [ ]:

def fit_fold(
    nombre,
    base_model,
    Xfit,
    yfit
):

    modelo = clone(
        base_model
    )

    if nombre == "xgb":

        freq = (
            pd.Series(yfit)
            .value_counts(
                normalize=True
            )
            .to_dict()
        )

        sw = np.array([
            1.0 / freq[v]
            for v in yfit
        ])

        modelo.fit(
            Xfit,
            yfit - 1,
            sample_weight=sw
        )

    else:

        modelo.fit(
            Xfit,
            yfit
        )

    return modelo


def generar_oof(
    nombre,
    base_model
):

    skf = StratifiedKFold(
        n_splits=N_FOLDS_OOF,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    oof = np.zeros(
        (len(Xtr), 4)
    )

    for fold, (
        idx_fit,
        idx_val
    ) in enumerate(
        skf.split(
            Xtr,
            ytr
        ),
        start=1
    ):

        print(
            f"{nombre} — fold "
            f"{fold}/{N_FOLDS_OOF}"
        )

        mod = fit_fold(
            nombre,
            base_model,
            Xtr.iloc[idx_fit],
            ytr[idx_fit]
        )

        oof[
            idx_val
        ] = proba_modelo(
            nombre,
            mod,
            Xtr.iloc[idx_val]
        )

    return normalizar_probabilidades(
        oof
    )


oof_raw = {}

for nombre, modelo in modelos.items():

    oof_raw[
        nombre
    ] = generar_oof(
        nombre,
        modelo
    )



## 15. Separar OOF en calibración-fit y selección

- 50% de OOF: ajustar Platt/Isotonic.
- 50% de OOF: seleccionar método y threshold.

El calibrador seleccionado **no se reajusta después**; así el threshold permanece en la misma escala probabilística con la que fue seleccionado.


In [ ]:

idx_cal_fit, idx_select = train_test_split(
    np.arange(len(ytr)),
    test_size=0.50,
    stratify=ytr,
    random_state=2025
)

print(
    "Calibration fit:",
    len(idx_cal_fit)
)

print(
    "Selection:",
    len(idx_select)
)

display(
    pd.DataFrame({
        "cal_fit": (
            pd.Series(
                ytr[idx_cal_fit]
            )
            .value_counts()
            .sort_index()
        ),
        "selection": (
            pd.Series(
                ytr[idx_select]
            )
            .value_counts()
            .sort_index()
        ),
    })
)


## 16. Platt e Isotonic one-vs-rest

In [ ]:

EPS = 1e-6


def ajustar_platt(
    p,
    ybin
):

    z = logit(
        np.clip(
            p,
            EPS,
            1-EPS
        )
    ).reshape(-1, 1)

    mod = LogisticRegression(
        solver="lbfgs",
        max_iter=2000,
        random_state=RANDOM_STATE
    )

    mod.fit(
        z,
        ybin
    )

    return mod


def aplicar_platt(
    mod,
    p
):

    z = logit(
        np.clip(
            p,
            EPS,
            1-EPS
        )
    ).reshape(-1, 1)

    return (
        mod.predict_proba(z)[:, 1]
    )


def ajustar_calibrador(
    metodo,
    p_fit,
    y_fit
):

    if metodo == "raw":
        return None

    mods = []

    for j, c in enumerate(CLASES):

        ybin = (
            y_fit == c
        ).astype(int)

        if metodo == "platt":

            mod = ajustar_platt(
                p_fit[:, j],
                ybin
            )

        elif metodo == "isotonic":

            mod = IsotonicRegression(
                y_min=0,
                y_max=1,
                out_of_bounds="clip"
            )

            mod.fit(
                p_fit[:, j],
                ybin
            )

        else:
            raise ValueError(
                metodo
            )

        mods.append(
            mod
        )

    return mods


def aplicar_calibrador(
    metodo,
    mods,
    p
):

    if metodo == "raw":

        return normalizar_probabilidades(
            p
        )

    out = np.zeros_like(
        p,
        dtype=float
    )

    for j in range(4):

        if metodo == "platt":

            out[:, j] = aplicar_platt(
                mods[j],
                p[:, j]
            )

        else:

            out[:, j] = mods[
                j
            ].predict(
                p[:, j]
            )

    return normalizar_probabilidades(
        out
    )


## 17. Seleccionar calibración por menor Brier Score

In [ ]:

calibradores = {}
calibracion_elegida = {}
probas_test_finales = {}
probas_select_finales = {}

filas_cal = []

for nombre in modelos:

    p_oof = oof_raw[
        nombre
    ]

    p_fit = p_oof[
        idx_cal_fit
    ]

    y_fit = ytr[
        idx_cal_fit
    ]

    p_select_raw = p_oof[
        idx_select
    ]

    y_select = ytr[
        idx_select
    ]

    candidatos = {}

    # RAW
    candidatos["raw"] = {
        "mods": None,
        "p": (
            normalizar_probabilidades(
                p_select_raw
            )
        ),
    }

    # PLATT
    mods_platt = ajustar_calibrador(
        "platt",
        p_fit,
        y_fit
    )

    candidatos["platt"] = {
        "mods": mods_platt,
        "p": aplicar_calibrador(
            "platt",
            mods_platt,
            p_select_raw
        ),
    }

    # ISOTONIC
    mods_iso = ajustar_calibrador(
        "isotonic",
        p_fit,
        y_fit
    )

    candidatos["isotonic"] = {
        "mods": mods_iso,
        "p": aplicar_calibrador(
            "isotonic",
            mods_iso,
            p_select_raw
        ),
    }

    evaluaciones = []

    for metodo, obj in candidatos.items():

        psel = obj[
            "p"
        ]

        fila = {
            "modelo": nombre,
            "metodo": metodo,

            "brier": brier_multiclase(
                y_select,
                psel
            ),

            "log_loss": log_loss(
                y_select,
                psel,
                labels=CLASES
            ),

            "ece": ece_confianza(
                y_select,
                psel
            ),

            "roc_auc_macro": roc_auc_score(
                y_select,
                psel,
                labels=CLASES,
                multi_class="ovr",
                average="macro"
            ),

            "average_precision_macro": (
                average_precision_score(
                    one_hot_y(
                        y_select
                    ),
                    psel,
                    average="macro"
                )
            ),
        }

        evaluaciones.append(
            fila
        )

        filas_cal.append(
            fila
        )

    tab = (
        pd.DataFrame(
            evaluaciones
        )
        .sort_values(
            [
                "brier",
                "log_loss"
            ]
        )
    )

    raw_brier = float(
        tab.loc[
            tab["metodo"] == "raw",
            "brier"
        ].iloc[0]
    )

    mejor = tab.iloc[0]

    mejora = (
        raw_brier
        - float(
            mejor["brier"]
        )
    )

    if (
        mejor["metodo"] != "raw"
        and mejora
        >= MIN_MEJORA_BRIER
    ):
        elegido = str(
            mejor["metodo"]
        )
    else:
        elegido = "raw"

    calibracion_elegida[
        nombre
    ] = elegido

    calibradores[
        nombre
    ] = candidatos[
        elegido
    ]["mods"]

    probas_select_finales[
        nombre
    ] = candidatos[
        elegido
    ]["p"]

    probas_test_finales[
        nombre
    ] = aplicar_calibrador(
        elegido,
        calibradores[
            nombre
        ],
        probas_test_raw[
            nombre
        ]
    )

    print(
        f"{nombre}: {elegido} | "
        f"mejora Brier={mejora:.6f}"
    )


tabla_calibracion = pd.DataFrame(
    filas_cal
)

display(
    tabla_calibracion.sort_values(
        [
            "modelo",
            "brier"
        ]
    )
)


## 18. Métricas TEST con probabilidades finales

In [ ]:

filas_finales = []

for nombre in modelos:

    m = metricas_globales(
        yte,
        probas_test_finales[
            nombre
        ]
    )

    m["modelo"] = nombre

    m["calibracion"] = (
        calibracion_elegida[
            nombre
        ]
    )

    filas_finales.append(
        m
    )


metricas_finales = (
    pd.DataFrame(
        filas_finales
    )
    .set_index("modelo")
)

display(
    metricas_finales
)


# PARTE IV — ROC y Precision–Recall

## 19. Curvas ROC one-vs-rest

In [ ]:

Yte = one_hot_y(
    yte
)

for nombre in modelos:

    p = probas_test_finales[
        nombre
    ]

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    for j, c in enumerate(CLASES):

        fpr, tpr, _ = roc_curve(
            Yte[:, j],
            p[:, j]
        )

        auc = roc_auc_score(
            Yte[:, j],
            p[:, j]
        )

        ax.plot(
            fpr,
            tpr,
            label=(
                f"C{c}: {NOMBRES[c]} "
                f"(AUC={auc:.3f})"
            )
        )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )

    ax.set_xlabel(
        "False Positive Rate"
    )

    ax.set_ylabel(
        "True Positive Rate"
    )

    ax.set_title(
        f"ROC one-vs-rest — {nombre.upper()}"
    )

    ax.legend(
        fontsize=8
    )

    ax.grid(
        alpha=0.2
    )

    plt.tight_layout()

    plt.savefig(
        FIG / f"roc_ovr_{nombre}.png",
        dpi=220,
        bbox_inches="tight"
    )

    plt.show()


## 20. Curvas Precision–Recall one-vs-rest

In [ ]:

for nombre in modelos:

    p = probas_test_finales[
        nombre
    ]

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    for j, c in enumerate(CLASES):

        prec, rec, _ = (
            precision_recall_curve(
                Yte[:, j],
                p[:, j]
            )
        )

        ap = average_precision_score(
            Yte[:, j],
            p[:, j]
        )

        baseline = (
            Yte[:, j]
            .mean()
        )

        ax.plot(
            rec,
            prec,
            label=(
                f"C{c}: {NOMBRES[c]} "
                f"(AP={ap:.3f}; "
                f"base={baseline:.3f})"
            )
        )

    ax.set_xlabel(
        "Recall"
    )

    ax.set_ylabel(
        "Precision"
    )

    ax.set_title(
        f"Precision–Recall — {nombre.upper()}"
    )

    ax.legend(
        fontsize=8
    )

    ax.grid(
        alpha=0.2
    )

    plt.tight_layout()

    plt.savefig(
        FIG / f"precision_recall_ovr_{nombre}.png",
        dpi=220,
        bbox_inches="tight"
    )

    plt.show()


## 21. Comparación directa del segmento 2

In [ ]:

fig, ax = plt.subplots(
    figsize=(8, 6)
)

for nombre in modelos:

    p = probas_test_finales[
        nombre
    ]

    prec, rec, _ = (
        precision_recall_curve(
            Yte[:, 1],
            p[:, 1]
        )
    )

    ap = average_precision_score(
        Yte[:, 1],
        p[:, 1]
    )

    ax.plot(
        rec,
        prec,
        label=(
            f"{nombre.upper()} "
            f"(AP={ap:.3f})"
        )
    )

ax.axhline(
    Yte[:, 1].mean(),
    linestyle="--",
    label=(
        f"Baseline="
        f"{Yte[:,1].mean():.3f}"
    )
)

ax.set_xlabel(
    "Recall"
)

ax.set_ylabel(
    "Precision"
)

ax.set_title(
    "Precision–Recall — Segmento 2"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

plt.savefig(
    FIG / "precision_recall_segmento2_modelos.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()


# PARTE V — Análisis de falsos negativos

In [ ]:

def analizar_fn(
    y_true,
    pred,
    modelo
):

    resumen = []
    destinos = []

    for c in CLASES:

        real = (
            y_true == c
        )

        fn = (
            real
            & (pred != c)
        )

        n_real = int(
            real.sum()
        )

        n_fn = int(
            fn.sum()
        )

        resumen.append({
            "modelo": modelo,
            "clase": c,
            "segmento": NOMBRES[c],
            "n_reales": n_real,
            "falsos_negativos": n_fn,
            "tasa_FN": (
                n_fn / n_real
                if n_real
                else np.nan
            ),
            "recall": (
                1 - n_fn / n_real
                if n_real
                else np.nan
            ),
        })

        destinos_c = (
            pd.Series(
                pred[fn]
            )
            .value_counts()
        )

        for destino, n in destinos_c.items():

            destinos.append({
                "modelo": modelo,
                "clase_real": c,
                "segmento_real": NOMBRES[c],
                "clase_destino": int(destino),
                "segmento_destino": (
                    NOMBRES[
                        int(destino)
                    ]
                ),
                "n": int(n),
                "pct_FN": (
                    n / n_fn
                    if n_fn
                    else np.nan
                ),
            })

    return (
        pd.DataFrame(resumen),
        pd.DataFrame(destinos)
    )


resumen_fn = []
destinos_fn = []

for nombre in modelos:

    pred = pred_argmax(
        probas_test_finales[
            nombre
        ]
    )

    r, d = analizar_fn(
        yte,
        pred,
        nombre
    )

    resumen_fn.append(
        r
    )

    destinos_fn.append(
        d
    )


tabla_fn = pd.concat(
    resumen_fn,
    ignore_index=True
)

tabla_destinos_fn = pd.concat(
    destinos_fn,
    ignore_index=True
)

print(
    "Falsos negativos por segmento"
)

display(
    tabla_fn
)

print(
    "Destino de los falsos negativos"
)

display(
    tabla_destinos_fn
)



# PARTE VI — Threshold específico del segmento 2

## 22. Regla de decisión

Para el análisis de sensibilidad:

\[
P(C_2)\geq t_2
\Rightarrow C_2
\]

En caso contrario:

\[
\widehat C
=
\arg\max\{
P(C_1),P(C_3),P(C_4)
\}
\]


In [ ]:

def pred_threshold_c2(
    proba,
    threshold
):

    pred = np.empty(
        len(proba),
        dtype=int
    )

    es_c2 = (
        proba[:, 1]
        >= threshold
    )

    pred[
        es_c2
    ] = 2

    otras = np.array(
        [1, 3, 4]
    )

    p_otras = proba[
        :,
        [0, 2, 3]
    ]

    pred[
        ~es_c2
    ] = otras[
        np.argmax(
            p_otras[
                ~es_c2
            ],
            axis=1
        )
    ]

    return pred


def tabla_threshold_c2(
    y_true,
    proba
):

    filas = []

    for t in np.arange(
        0.01,
        0.951,
        0.01
    ):

        pred = pred_threshold_c2(
            proba,
            t
        )

        y2 = (
            y_true == 2
        ).astype(int)

        p2 = (
            pred == 2
        ).astype(int)

        filas.append({
            "threshold_c2": t,

            "precision_c2": precision_score(
                y2,
                p2,
                zero_division=0
            ),

            "recall_c2": recall_score(
                y2,
                p2,
                zero_division=0
            ),

            "f1_c2": f1_score(
                y2,
                p2,
                zero_division=0
            ),

            "accuracy_global": accuracy_score(
                y_true,
                pred
            ),

            "f1_macro_global": f1_score(
                y_true,
                pred,
                labels=CLASES,
                average="macro",
                zero_division=0
            ),
        })

    return pd.DataFrame(
        filas
    )


## 23. Seleccionar threshold únicamente en Selection

In [ ]:

tablas_threshold = {}
decisiones = []

y_select = ytr[
    idx_select
]

for nombre in modelos:

    psel = probas_select_finales[
        nombre
    ]

    tabla = tabla_threshold_c2(
        y_select,
        psel
    )

    tablas_threshold[
        nombre
    ] = tabla

    fila_f1 = tabla.loc[
        tabla["f1_c2"].idxmax()
    ]

    cand70 = tabla.loc[
        tabla["recall_c2"]
        >= RECALL_OBJETIVO_C2
    ].copy()

    if len(
        cand70
    ):

        fila70 = (
            cand70
            .sort_values(
                [
                    "precision_c2",
                    "f1_c2"
                ],
                ascending=False
            )
            .iloc[0]
        )

    else:
        fila70 = None

    decisiones.append({
        "modelo": nombre,

        "threshold_f1_c2": (
            float(
                fila_f1[
                    "threshold_c2"
                ]
            )
        ),

        "precision_c2_selection": (
            float(
                fila_f1[
                    "precision_c2"
                ]
            )
        ),

        "recall_c2_selection": (
            float(
                fila_f1[
                    "recall_c2"
                ]
            )
        ),

        "f1_c2_selection": (
            float(
                fila_f1[
                    "f1_c2"
                ]
            )
        ),

        "threshold_recall70_c2": (
            float(
                fila70[
                    "threshold_c2"
                ]
            )
            if fila70 is not None
            else np.nan
        ),
    })


tabla_decisiones = pd.DataFrame(
    decisiones
)

display(
    tabla_decisiones
)


## 24. Aplicar los thresholds congelados a TEST

In [ ]:

filas_escenarios = []

for _, row in (
    tabla_decisiones
    .iterrows()
):

    nombre = row[
        "modelo"
    ]

    ptest = probas_test_finales[
        nombre
    ]

    escenarios = [
        (
            "argmax",
            None,
            pred_argmax(
                ptest
            )
        ),

        (
            "threshold_F1_C2",
            float(
                row[
                    "threshold_f1_c2"
                ]
            ),
            pred_threshold_c2(
                ptest,
                float(
                    row[
                        "threshold_f1_c2"
                    ]
                )
            )
        ),
    ]

    if not pd.isna(
        row[
            "threshold_recall70_c2"
        ]
    ):

        t70 = float(
            row[
                "threshold_recall70_c2"
            ]
        )

        escenarios.append(
            (
                "threshold_Recall70_C2",
                t70,
                pred_threshold_c2(
                    ptest,
                    t70
                )
            )
        )

    for escenario, t, pred in escenarios:

        y2 = (
            yte == 2
        ).astype(int)

        p2 = (
            pred == 2
        ).astype(int)

        filas_escenarios.append({
            "modelo": nombre,
            "escenario": escenario,
            "threshold_c2": t,

            "accuracy": accuracy_score(
                yte,
                pred
            ),

            "precision_macro": precision_score(
                yte,
                pred,
                labels=CLASES,
                average="macro",
                zero_division=0
            ),

            "recall_macro": recall_score(
                yte,
                pred,
                labels=CLASES,
                average="macro",
                zero_division=0
            ),

            "f1_macro": f1_score(
                yte,
                pred,
                labels=CLASES,
                average="macro",
                zero_division=0
            ),

            "precision_c2": precision_score(
                y2,
                p2,
                zero_division=0
            ),

            "recall_c2": recall_score(
                y2,
                p2,
                zero_division=0
            ),

            "f1_c2": f1_score(
                y2,
                p2,
                zero_division=0
            ),
        })


tabla_escenarios = pd.DataFrame(
    filas_escenarios
)

display(
    tabla_escenarios
)


## 25. Sensibilidad del F1 del segmento 2

In [ ]:

fig, ax = plt.subplots(
    figsize=(9, 6)
)

for nombre, tabla in tablas_threshold.items():

    ax.plot(
        tabla["threshold_c2"],
        tabla["f1_c2"],
        label=nombre.upper()
    )

ax.set_xlabel(
    "Threshold segmento 2"
)

ax.set_ylabel(
    "F1 segmento 2"
)

ax.set_title(
    "Sensibilidad del F1 de C2 al threshold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

plt.savefig(
    FIG / "sensibilidad_threshold_C2.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()


## 26. Matrices de confusión argmax

In [ ]:

matrices = []

for nombre in modelos:

    pred = pred_argmax(
        probas_test_finales[
            nombre
        ]
    )

    cm = confusion_matrix(
        yte,
        pred,
        labels=CLASES
    )

    cm_norm = confusion_matrix(
        yte,
        pred,
        labels=CLASES,
        normalize="true"
    )

    fig, ax = plt.subplots(
        figsize=(6, 5)
    )

    im = ax.imshow(
        cm_norm
    )

    ax.set_xticks(
        range(4)
    )

    ax.set_yticks(
        range(4)
    )

    ax.set_xticklabels(
        CLASES
    )

    ax.set_yticklabels(
        CLASES
    )

    ax.set_xlabel(
        "Predicción"
    )

    ax.set_ylabel(
        "Real"
    )

    ax.set_title(
        f"Matriz de confusión — {nombre.upper()}"
    )

    for i in range(4):
        for j in range(4):

            ax.text(
                j,
                i,
                (
                    f"{cm_norm[i,j]:.1%}\n"
                    f"{cm[i,j]}"
                ),
                ha="center",
                va="center"
            )

            matrices.append({
                "modelo": nombre,
                "real": int(
                    CLASES[i]
                ),
                "predicho": int(
                    CLASES[j]
                ),
                "n": int(
                    cm[i,j]
                ),
                "proporcion_fila": float(
                    cm_norm[i,j]
                ),
            })

    plt.colorbar(
        im,
        ax=ax
    )

    plt.tight_layout()

    plt.savefig(
        FIG / f"matriz_confusion_{nombre}.png",
        dpi=220,
        bbox_inches="tight"
    )

    plt.show()


tabla_matrices = pd.DataFrame(
    matrices
)


## 27. Curvas de calibración one-vs-rest

In [ ]:

def bins_calibracion(
    ybin,
    p,
    q=10
):

    temp = pd.DataFrame({
        "y": ybin,
        "p": p
    })

    temp["bin"] = pd.qcut(
        temp["p"],
        q=q,
        duplicates="drop"
    )

    return (
        temp.groupby(
            "bin",
            observed=True
        )
        .agg(
            prob_media=(
                "p",
                "mean"
            ),
            frecuencia_observada=(
                "y",
                "mean"
            ),
            n=(
                "y",
                "size"
            ),
        )
        .reset_index(
            drop=True
        )
    )


for nombre in modelos:

    p = probas_test_finales[
        nombre
    ]

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(11, 9)
    )

    for j, c in enumerate(CLASES):

        ax = axes[
            j // 2
        ][
            j % 2
        ]

        tabla_c = bins_calibracion(
            Yte[:, j],
            p[:, j]
        )

        ax.plot(
            tabla_c[
                "prob_media"
            ],
            tabla_c[
                "frecuencia_observada"
            ],
            marker="o"
        )

        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--"
        )

        ax.set_title(
            f"C{c}: {NOMBRES[c]}"
        )

        ax.set_xlabel(
            "Probabilidad predicha"
        )

        ax.set_ylabel(
            "Frecuencia observada"
        )

        ax.grid(
            alpha=0.2
        )

    fig.suptitle(
        (
            f"Calibración — "
            f"{nombre.upper()} — "
            f"{calibracion_elegida[nombre]}"
        ),
        fontsize=13
    )

    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.96
        ]
    )

    plt.savefig(
        FIG / f"calibracion_{nombre}.png",
        dpi=220,
        bbox_inches="tight"
    )

    plt.show()



# PARTE VII — Corrección opcional del beeswarm SHAP

El ZIP que subiste sí contiene `shap_values_xgb.npz`.

La siguiente sección utiliza los **índices globales exactos guardados por `06_shap.py`**, evitando mezclar valores SHAP de unas observaciones con features de otras.

Déjala activada si quieres regenerar la figura.


In [ ]:

EJECUTAR_SHAP_CORREGIDO = True

if EJECUTAR_SHAP_CORREGIDO:

    %pip install -q shap
    import shap

    if SHAP_NPZ is None:

        print(
            "No existe shap_values_xgb.npz."
        )

    else:

        z = np.load(
            SHAP_NPZ,
            allow_pickle=True
        )

        sv = z[
            "sv"
        ]

        idx_global = np.asarray(
            z[
                "idx_test"
            ],
            dtype=int
        )

        feat_names = list(
            z[
                "features"
            ]
        )

        X_sh = X.iloc[
            idx_global
        ].copy()

        if (
            list(
                X_sh.columns
            )
            != feat_names
        ):
            raise RuntimeError(
                "Las features no coinciden "
                "con las almacenadas por SHAP."
            )

        rng_shap = (
            np.random.RandomState(
                7
            )
        )

        n_plot = min(
            600,
            sv.shape[1]
        )

        idx_plot = rng_shap.choice(
            sv.shape[1],
            size=n_plot,
            replace=False
        )

        X_plot = X_sh.iloc[
            idx_plot
        ]

        fig, axes = plt.subplots(
            2,
            2,
            figsize=(16, 13)
        )

        for i, c in enumerate(
            CLASES
        ):

            ax = axes[
                i // 2
            ][
                i % 2
            ]

            plt.sca(
                ax
            )

            shap.summary_plot(
                sv[c - 1][idx_plot],
                X_plot,
                feature_names=feat_names,
                show=False,
                max_display=12,
                plot_size=None
            )

            ax.set_title(
                f"C{c}: {NOMBRES[c]}"
            )

        fig.suptitle(
            "SHAP beeswarm corregido",
            fontsize=13
        )

        plt.tight_layout(
            rect=[
                0,
                0,
                1,
                0.97
            ]
        )

        plt.savefig(
            FIG / "shap_beeswarm_corregido.png",
            dpi=220,
            bbox_inches="tight"
        )

        plt.show()


# PARTE VIII — Exportar todos los resultados

In [ ]:

# TABLAS
metricas_raw.to_csv(
    TAB / "01_metricas_RAW_test.csv",
    encoding="utf-8-sig"
)

metricas_raw_ponderadas.to_csv(
    TAB / "02_metricas_RAW_test_ponderadas_fexp.csv",
    encoding="utf-8-sig"
)

comparacion_original.to_csv(
    TAB / "03_comparacion_con_resultados_originales.csv",
    index=False,
    encoding="utf-8-sig"
)

metricas_por_segmento.to_csv(
    TAB / "04_metricas_por_segmento.csv",
    index=False,
    encoding="utf-8-sig"
)

tabla_calibracion.to_csv(
    TAB / "05_seleccion_calibracion.csv",
    index=False,
    encoding="utf-8-sig"
)

metricas_finales.to_csv(
    TAB / "06_metricas_finales_TEST.csv",
    encoding="utf-8-sig"
)

tabla_fn.to_csv(
    TAB / "07_falsos_negativos.csv",
    index=False,
    encoding="utf-8-sig"
)

tabla_destinos_fn.to_csv(
    TAB / "08_destino_falsos_negativos.csv",
    index=False,
    encoding="utf-8-sig"
)

tabla_decisiones.to_csv(
    TAB / "09_thresholds_seleccionados_C2.csv",
    index=False,
    encoding="utf-8-sig"
)

tabla_escenarios.to_csv(
    TAB / "10_escenarios_threshold_C2_TEST.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.concat(
    [
        tabla.assign(
            modelo=nombre
        )
        for nombre, tabla
        in tablas_threshold.items()
    ],
    ignore_index=True
).to_csv(
    TAB / "11_sensibilidad_threshold_C2.csv",
    index=False,
    encoding="utf-8-sig"
)

tabla_matrices.to_csv(
    TAB / "12_matrices_confusion.csv",
    index=False,
    encoding="utf-8-sig"
)


# METADATOS
metadata = {
    "split_original_reutilizado":
        SPLIT_FILE is not None,

    "n_train":
        int(len(Xtr)),

    "n_test":
        int(len(Xte)),

    "calibracion_elegida":
        calibracion_elegida,

    "recall_objetivo_c2":
        RECALL_OBJETIVO_C2,

    "random_state":
        RANDOM_STATE,

    "rf_best_params":
        rf_search.best_params_,

    "nota":
        (
            "Argmax es la regla principal. "
            "Threshold C2 es análisis de sensibilidad."
        ),
}

with open(
    TAB / "13_metadatos.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# README
readme = '''
ENEMDU 2025 - METRICAS AVANZADAS MULTICLASE

Este paquete contiene la reproducción de:
- Logit multinomial
- Random Forest
- XGBoost

y las ampliaciones:
- Brier multiclase
- calibración RAW / Platt / Isotonic
- ROC one-vs-rest
- Precision-Recall one-vs-rest
- Average Precision
- falsos negativos
- threshold específico para segmento 2
- escenario Recall C2 >= 70%

Archivos centrales:
01_metricas_RAW_test.csv
04_metricas_por_segmento.csv
05_seleccion_calibracion.csv
06_metricas_finales_TEST.csv
07_falsos_negativos.csv
08_destino_falsos_negativos.csv
09_thresholds_seleccionados_C2.csv
10_escenarios_threshold_C2_TEST.csv
'''

with open(
    RESULTADOS / "00_LEEME.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        readme
    )


# ZIP
ZIP_RESULTADOS = (
    "/content/resultados_avanzados_multiclase"
)

zip_out = shutil.make_archive(
    ZIP_RESULTADOS,
    "zip",
    root_dir=RESULTADOS
)

print(
    "ZIP final:",
    zip_out
)

from google.colab import files

files.download(
    zip_out
)



# Qué debes enviarme

Cuando termine el notebook, Colab descargará:

**`resultados_avanzados_multiclase.zip`**

Ese es el único archivo que necesito que me subas.

Con ese ZIP podremos revisar:

- si se reprodujeron los resultados originales;
- cuál modelo gana con ROC-AUC y AP;
- qué tan mala es realmente la clase 2;
- hacia qué segmento se van sus falsos negativos;
- si Platt o Isotonic aportan;
- qué ocurre al buscar Recall ≥70%;
- qué resultados deben entrar en el artículo y cuáles deben ir al anexo.
